# Product entity matching: data, features and evaluation

**English** | [简体中文](product-matching-walkthrough.zh-CN.ipynb)

Two retailers may write different titles for the same product. This study compares pairs of listings using text and field comparisons. Follow the saved results, then run small examples to see how the inputs and metrics work.

The final model is named **C+** in this project: RoBERTa reads the paired product text, combines it with 42 field comparisons, and averages match probabilities across three training seeds. The comparison table below defines each method.

The measured results concern **555 fixed Walmart–Amazon pairs**. This is a retrospective comparison: historical exposure is incompletely documented, and performance on independent new Walmart–Amazon samples has not been tested. The Acme listings, labels and probabilities below are invented teaching examples, not model predictions.

**Run it:** use a Python 3.11+ kernel and run the cells in order. From the repository root, `python scripts/run_notebook.py` checks both translations against their saved outputs using only the standard library. It does not download data or models, install packages, or train a model.

[Project overview](../README.md) · [Data and results](../showcase/en/index.html) · [Reproduction guide](../docs/REPRODUCIBILITY.md)


In [1]:
from pathlib import Path
import json, math, sys
LANGUAGE = "en"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "results/fixed-results.json").is_file() and (p / "src/product_matching").is_dir())
sys.path.insert(0, str(ROOT / "src"))
facts = json.loads((ROOT / "results/fixed-results.json").read_text(encoding="utf-8"))
text = json.loads((ROOT / "notebooks/walkthrough_text.json").read_text(encoding="utf-8"))[LANGUAGE]
print(text["task"])
for role, count in facts["roles"].items():
    print(f'{text["roles"][role]}: {count:,}')
print(text["scope"])


Task: decide whether two retailer listings refer to the same product.
Fit pairs: 3,738
Development pairs: 199
Calibration pairs: 209
Evaluation pairs: 555
Scope: retrospective results; historical exposure is incompletely documented; independent new Walmart–Amazon sample performance is untested.


## 1. Data roles and label separation

| Role | Pairs | Use |
| --- | ---: | --- |
| Fit | 3,738 | Update model parameters and fit title evidence |
| Development | 199 | Select a shared checkpoint epoch |
| Calibration | 209 | Select a classification cutoff |
| Evaluation | 555 | Measure the frozen models and cutoffs |

A `pair_id` aligns a prediction with its label; row order is not treated as identity. Only **fit model inputs** carry target labels. Calibration and evaluation labels are separate inputs to the corresponding metric steps.

The pipeline is `fit-only IDF → role preparation → development selection → calibration cutoff → evaluation`. These role names describe intended use. They do not establish that the historical evaluation pool was untouched.


## 2. Fixed-pool model comparison

This table loads the public aggregate results, not the synthetic examples that appear later. All six methods used the same 555-pair pool. B22, L30, S33 and L42 are the reference methods; C+ is the final neural ensemble, and C0 is a separately trained ablation network whose 42 comparison features are zero throughout training and inference.

Read **AP** as ranking quality, **F1** as the balance of false and missed matches at the calibration cutoff, and **P@100** as precision among the first 100 ranked pairs. AP is not classification accuracy.


In [2]:
print(f'{text["method"]:<8} {"AP":>9} {"F1":>9} {"P@100":>9}')
for method, result in facts["metrics"].items():
    print(f'{method:<8} {result["ap"]:9.6f} {result["f1"]:9.6f} {result["p_at_100"]:9.2f}')


Method          AP        F1     P@100
B22       0.795590  0.743719      0.81
L30       0.844899  0.762115      0.90
S33       0.846702  0.767123      0.89
L42       0.949700  0.852041      1.00
C+        0.965998  0.901639      1.00
C0        0.965616  0.875648      1.00


### Absolute differences in percentage points

L42 is the strongest tabular reference in this pool. Subtract its score from C+, then multiply by 100 to express an absolute difference in percentage points. Compare separately trained C0 as well: its feature input is zero throughout both training and inference.


In [3]:
cplus, reference, ablation = (facts["metrics"][m] for m in ("C+", "L42", "C0"))
print(f'{text["ap_gain"]}: {(cplus["ap"]-reference["ap"])*100:.4f} {text["pp"]}')
print(f'{text["f1_gain"]}: {(cplus["f1"]-reference["f1"])*100:.4f} {text["pp"]}')
print(f'{text["ap_c0"]}: {(cplus["ap"]-ablation["ap"])*100:.4f} {text["pp"]}')


AP gain over L42: 1.6298 percentage points
F1 gain over L42: 4.9599 percentage points
AP difference versus C0: 0.0381 percentage points


C+ is ahead of L42, but its AP is close to C0. These numbers do not establish a large AP contribution from the 42 features or a statistically significant effect. They describe this fixed pool; they are not a new-source generalization result.


## 3. Feature construction for a synthetic product pair

The invented Acme listings differ in model-number punctuation and title spacing. The feature vector combines **14 business comparisons + 8 title comparisons (4 fit-only IDF-weighted comparisons and 4 direct comparisons) + 8 title-code comparisons + 12 native model/category comparisons**. The output shows four of those fields, not a match prediction.

The prices below can be compared because both parse successfully and both use USD. A missing price stays missing; an unknown or different currency makes the price comparison unavailable. Compact strings help compare `ZX-100` with `ZX100`, but a similar-looking model such as `ZX-101` must not be silently treated as equivalent.

The toy fit label only exercises the input schema and IDF construction. **No classifier is trained, and RoBERTa does not score these records.** See the [code map](../docs/CODE_MAP.md) for the feature implementation.


In [4]:
from product_matching.data import business_record, native_record
from product_matching.features import FEATURE_NAMES_42, fit_title_evidence, prepare_rows
raw_left = {"title":"Acme ZX-100 16GB", "brand":"Acme", "price":"19.99", "priceCurrency":"USD"}
raw_right = {"title":"ACME ZX100 16 GB", "brand":"ACME", "price":"20.00", "priceCurrency":"USD"}
example = {"pair_id":"synthetic:acme", "left":business_record(raw_left), "right":business_record(raw_right),
           "left_native":native_record({"modelno":"ZX-100", "category":"electronics"}),
           "right_native":native_record({"modelno":"ZX100", "category":"electronics"})}
fit_idf = fit_title_evidence([{**example, "label":1}])
prepared = prepare_rows([example], role="dev", fit_idf=fit_idf)
values = dict(zip(FEATURE_NAMES_42, prepared[0]["x42"]))
print(text["feature_width"], ":", len(values))
for key in ("brand_equal", "price_comparable", "title_compact_exact", "native_model_compact_exact"):
    print(f'{text["features"][key]} [{key}]: {values[key]:.6f}')
assert len(values)==42 and "label" not in prepared[0]


Synthetic feature count : 42
Brand agreement [brand_equal]: 1.000000
Comparable prices [price_comparable]: 1.000000
Equal compact titles [title_compact_exact]: 1.000000
Equal compact model numbers [native_model_compact_exact]: 1.000000


### Input-role validation

Try adding a target label to an evaluation input. The interface should reject it and leave the fit-only IDF state unchanged. Passing this check demonstrates the current interface rule; it cannot retroactively prove that all historical data use was leakage-free.


In [5]:
from product_matching.errors import AdmissionError
before = fit_idf.payload()
try:
    prepare_rows([{**example, "label":1}], role="evaluation", fit_idf=fit_idf)
except AdmissionError:
    print(text["label_rejected"])
else:
    raise AssertionError(text["label_accepted"])
assert before == fit_idf.payload()
print(text["idf_unchanged"])


The evaluation input rejected a target label, as expected.
The fit-only IDF state is unchanged.


## 4. AB/BA directions and three-seed aggregation

RoBERTa-base reads the pair in both AB and BA order. A 64-unit projection is concatenated with 42 comparison features; another 64-unit GELU layer, dropout and a scalar output produce a logit. For each seed, the AB/BA logits are averaged before sigmoid. The three seed probabilities are then averaged equally.

C+ follows a six-epoch training schedule; development data selects the shared epoch-4 checkpoints for the ensemble. All three seeds remain in the ensemble. Training used BF16 encoder autocast with FP32 parameters and head; inference used FP32. See the [model card](../docs/MODEL_CARD.md) for the fixed configuration.

The next cell uses **invented probabilities 0.9, 0.3 and 0.3** to explain the arithmetic. They are not RoBERTa outputs or measured seed results.


In [6]:
from product_matching.metrics import mean_scores
invented = {seed:[{"pair_id":"synthetic:acme", "score":value}]
            for seed,value in zip((42,43,44),(.9,.3,.3))}
mean = mean_scores(["synthetic:acme"], invented)[0]["score"]
print(text["mean"], ":", mean)
print(text["seeds"], ":", facts["model"]["seeds"], ";", text["epoch"], ":", facts["model"]["epoch"])
assert mean == .5


Invented probabilities: equal mean : 0.5
Study seeds : [42, 43, 44] ; Shared epoch : 4


## 5. Ranking metrics, cutoff and confusion counts

AP combines precision and recall changes along the ranking. Equal scores are handled together, so it does not award an arbitrary advantage to whichever tied item happens to appear first. In the invented example below, the top tie contains one match and one non-match. Its AP is 0.583333; that is not a classification accuracy.

The real experiment selected its cutoff on calibration data by F1, breaking ties by precision and then the higher cutoff. Evaluation did not choose a new cutoff. After the synthetic AP example, the cell reads C+'s **saved aggregate confusion counts** and independently recovers F1 as `2 × TP / (2 × TP + FP + FN)`.

Here, TP means found matches, FP means false matches, FN means missed matches, and TN means correctly rejected non-matches. Changing the cutoff can change these counts and F1; it does not change the underlying ranking AP.


In [7]:
from product_matching.metrics import average_precision
toy = [{"pair_id":"synthetic:a", "label":1, "score":.9},
       {"pair_id":"synthetic:b", "label":0, "score":.9},
       {"pair_id":"synthetic:c", "label":1, "score":.5}]
print(f'{text["toy_ap"]}: {average_precision(toy):.6f}')
print(text["threshold"], ":", cplus["threshold"])
confusion = cplus["confusion"]
tp,fp,fn,tn = (confusion[k] for k in ("tp","fp","fn","tn"))
f1_from_counts = 2*tp/(2*tp+fp+fn)
assert math.isclose(f1_from_counts, cplus["f1"], rel_tol=0, abs_tol=1e-14)
print(f'{text["confusion"]}: TP={tp}, FP={fp}, FN={fn}, TN={tn}')
print(f'{text["f1_recovered"]}: {f1_from_counts:.6f}')


Invented scores: AP with a tie: 0.583333
Saved C+ calibration cutoff : 0.3947772259513537
Saved aggregate counts: TP=165, FP=11, FN=25, TN=354
F1 recovered from the saved counts: 0.901639


## 6. Reproduction scope and execution commands

| Check | What the evidence covers |
| --- | --- |
| Public smoke checks and this Notebook | Aggregate arithmetic, input rules and synthetic examples; no weights or product dataset needed |
| Existing-weight replay | 20 commands and 23 result groups on a Windows RTX 3060 laptop matched the original outputs, with zero prediction difference |
| Raw-CSV reconstruction and full retraining | Not yet validated end to end |

The replay used locally supplied original artifacts and rebuilt model inputs. It confirms that the new code entry can replay existing models. It did not train new models or use new evaluation labels. The public repository does not distribute product records or trained weights.

Run `python scripts/public_smoke.py` and `python scripts/run_notebook.py` for the lightweight checks. Read the [reproduction guide](../docs/REPRODUCIBILITY.md) for the separate model-running commands and the [limitations](../docs/LIMITATIONS.md) for interpretation. This Notebook does not invoke those GPU commands.


In [8]:
for key, value in facts["reproduction"].items():
    display = text["yes"] if value is True else text["no"] if value is False else value
    print(f'{text["evidence"][key]}: {display}')
heavy = {"torch", "transformers", "sklearn", "sentence_transformers"}
assert not heavy & set(sys.modules)
print(text["done"])


Core tests recorded with the replay: 48
Commands in the existing-weight replay: 20
Independently checked result groups: 23
Maximum prediction difference: 0.0
New training during the replay: no
Full retraining validated: no
Completed: public aggregates and synthetic examples only; no model libraries, product dataset, inference or training.


## Understanding checks

1. **Does AP 0.9660 mean 96.60% classification accuracy, or P@100 = 1 mean every pair is correct?** Neither. AP measures ranking; P@100 covers only the first 100 ranked pairs.
2. **Did the Acme example run RoBERTa, or did the existing-weight replay repeat training?** No. Acme illustrates input processing; replay loads already trained parameters.
3. **Can evaluation labels choose a better cutoff?** No. Select it on calibration data and freeze it before evaluation.

Continue with the [project card](../docs/PROJECT_CARD.md) or the [model and evidence limitations](../docs/LIMITATIONS.md).
